In [1]:
from pyspark.sql.functions import *

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 3, Finished, Available, Finished, False)

In [2]:
bronze_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/bronze/orders"
silver_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/silver/orders"

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 4, Finished, Available, Finished, False)

In [3]:
df_bronze = spark.readStream.format("delta").load(bronze_path)

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 5, Finished, Available, Finished, False)

In [4]:
# Clean and Enrich
df_clean = (
    df_bronze
        .withColumn("timestamp", to_timestamp("timestamp"))
        .withWatermark("timestamp", "1 minute")
        .withColumn("price", when(col("price").isNull(), 0.0).otherwise(col("price")))
        .withColumn("quantity", when(col("quantity").isNull(), 1).otherwise(col("quantity")))
        .withColumn("total_amount", col("price") * col("quantity"))
        .dropDuplicates(["order_id", "timestamp"])
        .filter(col("country") == "USA")
        .filter(col("state").isNotNull())
)

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 6, Finished, Available, Finished, False)

In [5]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 7, Finished, Available, Finished, False)

DataFrame[]

In [6]:
# write to silver layer
silver_checkpoint = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Files/_checkpoints/silver_orders"

query = (
    df_clean.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", silver_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("silver.orders")
)
query.awaitTermination()

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 8, Finished, Available, Finished, False)

In [7]:
df_silver = spark.read.format("delta").load(silver_path)
print("Silver count:", df_silver.count())
display(df_silver)

StatementMeta(, 80cf26b2-96d7-4f3d-b160-0245e89072c1, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 650899fe-aa7f-423f-8e2f-f5740a040e3f)